# fGPU H2 상보 워크로드 공유 이득 실험

**H2 가설**: 한쪽이 idle-heavy(메모리 상주 + 간헐 연산), 다른 쪽이 compute-heavy(LLM generate)일 때,
idle 구간이 SM 경합을 줄여 **공유 이득(speedup > 1 또는 tps 유지)** 이 나온다.

- **A 세션** (`WORKLOAD="compute"`) — Qwen2-0.5B-Instruct LLM generate, ratio≈0.5
- **B 세션** (`WORKLOAD="memory_idle"`) — 큰 텐서 상주 + 주기적 matmul burst + sleep, ratio≈0.5

**throttle**: H2는 throttle **OFF** 필수. `compute_ratio` 미포함 세션으로 생성 → hook 기본값 OFF.
노트북 실행 전 아래 환경 확인 셀에서 `FGPU_THROTTLE_ENABLE=<unset>` 을 반드시 검증하세요.

**결과 파일**: `/workspace/session_result_{SCENARIO}.csv`, `/workspace/session_meta_{SCENARIO}.json`

> **H1과의 차이**: H1은 throttle ON + 양쪽 compute-heavy. H2는 throttle OFF + 비대칭 워크로드.
> 두 실험은 서로 다른 축을 측정합니다.

In [ ]:
# ══════════════════════════════════════════════════════
# CONFIG — 실험 전 여기만 수정
# ══════════════════════════════════════════════════════

# ─── 워크로드 종류 ──────────────────────────────────────
# "compute"      : LLM generate (A 세션용)
# "memory_idle"  : 큰 텐서 상주 + matmul burst + sleep (B 세션용)
WORKLOAD = "compute"

# ─── 세션 식별자 ─────────────────────────────────────────
SESSION_LABEL = "A"          # "A" 또는 "B"

# ─── 시나리오 ─────────────────────────────────────────────
# "solo"         : 단독 기준선
# "h2_sharing"   : H2 동시 공유 실험
SCENARIO = "solo"

# ─── 메모리 비율 (기록용, 실제 quota 는 세션 spawn 시 주입) ──
RATIO = 0.5

# ─── compute 워크로드 파라미터 (WORKLOAD=="compute" 시 사용) ─
MODEL_ID        = "Qwen/Qwen2-0.5B-Instruct"
N_ITERS         = 20
MAX_NEW_TOKENS  = 128
WARMUP          = 3
PROMPT = (
    "Explain the concept of fractional GPU resource sharing in cloud computing. "
    "Describe how multiple workloads can share a single GPU efficiently."
)

# ─── memory_idle 워크로드 파라미터 (WORKLOAD=="memory_idle" 시 사용) ─
# RESIDENT_GIB  : GPU에 상주시킬 큰 텐서 크기 (float32 기준 실제 MiB = GIB×1024)
# MATMUL_DIM    : iter마다 1회 실행할 정사각 matmul 차원 (N×N)
# IDLE_SLEEP_S  : matmul 후 다음 iter 전 sleep 길이 (초)
#   idle fraction ≈ IDLE_SLEEP_S / (matmul_time + IDLE_SLEEP_S)
#   MATMUL_DIM=4096 matmul ≈ 10~50ms → IDLE_SLEEP_S=8 → idle ≈ 99%
RESIDENT_GIB  = 4.0
MATMUL_DIM    = 4096
IDLE_SLEEP_S  = 8.0
N_ITERS_IDLE  = 20    # memory_idle 루프 횟수 (N_ITERS와 독립 관리)

# ─── 공통 파라미터 ────────────────────────────────────────
POLL_INTERVAL  = 0.05   # pynvml 폴링 간격 (초, 50ms)
MUTE_FGPU_LOG  = True   # [fgpu] ALLOW/FREE 로그 음소거 (렉 방지)


In [ ]:
# ══════════════════════════════════════════════════════
# 설치 — 멱등 (이미 깔려 있으면 무해)
# ══════════════════════════════════════════════════════
import sys, subprocess, os

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers", "accelerate", "nvidia-ml-py"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("[h2-infer] pip install 실패:", result.stderr[:500])
else:
    print("[h2-infer] 패키지 설치/확인 완료")

os.environ["HF_HOME"] = "/workspace/hf_cache"
os.makedirs("/workspace/hf_cache", exist_ok=True)
print(f"[h2-infer] HF_HOME = {os.environ['HF_HOME']}")


In [ ]:
# ══════════════════════════════════════════════════════
# 환경 출력 — hook / quota / throttle 컨텍스트 확인
# ══════════════════════════════════════════════════════
# ★ H2 핵심 검증: throttle이 실제로 OFF인지 여기서 확인한다.
#   FGPU_THROTTLE_ENABLE 가 <unset> 이어야 정상.
#   ON이면 H2 측정이 throttle 간섭으로 오염됨 → 세션 재생성 필요.
# ══════════════════════════════════════════════════════
import os
import torch

print("=" * 64)
print(f"  SESSION_LABEL  : {SESSION_LABEL}")
print(f"  SCENARIO       : {SCENARIO}")
print(f"  WORKLOAD       : {WORKLOAD}")
print(f"  RATIO (기록용) : {RATIO}")
print("-" * 64)

cuda_ok = torch.cuda.is_available()
print(f"  torch.cuda.is_available() : {cuda_ok}")
if cuda_ok:
    print(f"  GPU 디바이스              : {torch.cuda.get_device_name(0)}")
    total_mib = torch.cuda.get_device_properties(0).total_memory / 1024**2
    print(f"  GPU 총 메모리 (MiB)       : {total_mib:.0f}")
    quota_mib = total_mib * RATIO
    print(f"  예상 quota @ RATIO={RATIO}  : {quota_mib:.0f} MiB")
else:
    print("  [경고] CUDA 없음 — '--gpus all' 누락 또는 hook 미주입 의심")

print("-" * 64)

fgpu_vars = [
    "PYTORCH_NO_CUDA_MEMORY_CACHING",
    "LD_PRELOAD",
    "FGPU_RATIO",
    "FGPU_QUOTA_BYTES",
    "FGPU_THROTTLE_ENABLE",
    "FGPU_COMPUTE_RATIO",
    "FGPU_LAUNCH_LOG_EVERY",
]
for var in fgpu_vars:
    val = os.environ.get(var, "<unset>")
    print(f"  {var:<40}: {val}")

print("-" * 64)

# ★ throttle 상태 검증 (H2 핵심)
throttle_enable = os.environ.get("FGPU_THROTTLE_ENABLE", "")
compute_ratio_env = os.environ.get("FGPU_COMPUTE_RATIO", "")

if throttle_enable == "1" or compute_ratio_env:
    print()
    print("  ╔══════════════════════════════════════════════════════════╗")
    print("  ║  ⚠ 경고: throttle이 ON 상태입니다!                     ║")
    print("  ║                                                          ║")
    print(f"  ║  FGPU_THROTTLE_ENABLE = {throttle_enable!r:<36}║")
    print(f"  ║  FGPU_COMPUTE_RATIO   = {compute_ratio_env!r:<36}║")
    print("  ║                                                          ║")
    print("  ║  H2는 throttle OFF 세션으로 만들어야 합니다.            ║")
    print("  ║  세션 생성 시 compute_ratio 를 생략하세요:              ║")
    print('  ║  {"ratio":0.5,"mode":"jupyter","image":"fgpu-..."}      ║')
    print("  ║  (compute_ratio 미포함 → FGPU_THROTTLE_ENABLE 미주입)  ║")
    print("  ║  현재 세션을 삭제하고 throttle OFF 세션으로 재생성하세요.║")
    print("  ╚══════════════════════════════════════════════════════════╝")
else:
    print()
    print("  ✓ throttle OFF 확인: FGPU_THROTTLE_ENABLE=<unset>")
    print("  H2 측정에 적합한 세션입니다.")

print("=" * 64)


In [ ]:
# ══════════════════════════════════════════════════════
# pynvml GPU 폴링 유틸 — 배경 스레드
# ══════════════════════════════════════════════════════
# memory_idle 워크로드에서 burst/sleep 골이 gpu_util 타임라인에
# 뚜렷이 찍히는지 확인하는 것이 H2의 핵심 시각적 증거다.
# H2의 B 세션에서 sleep 구간 util≈0 이 관측되면 idle fraction이 확인됨.
# 주의: pynvml PID는 호스트 namespace → per-process 분해 불가.
#       GPU 전체 util/mem 을 두 컨테이너 합산 보조 proxy 로만 사용.

import threading
import time as _time

try:
    import pynvml
    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    PYNVML_OK = True
    print("[h2-infer] pynvml 초기화 완료")
except Exception as e:
    PYNVML_OK = False
    print(f"[h2-infer] pynvml 비활성: {e}")
    print("           GPU util 시계열은 기록되지 않습니다.")


class GpuPoller:
    """측정 루프 실행 중 GPU util% 와 used mem(MiB) 을 백그라운드 폴링.

    H2 memory_idle 세션에서 sleep 구간 util≈0 → burst 구간 spike 패턴이
    관측되면 idle fraction이 실험 파라미터로 제어됨을 확인할 수 있다.
    """

    def __init__(self, interval: float = 0.05):
        self.interval = interval
        self._stop = threading.Event()
        self.samples: list[tuple[float, int, float]] = []
        self._thread = threading.Thread(target=self._run, daemon=True)

    def start(self):
        self._stop.clear()
        self.samples.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self, timeout: float = 2.0):
        self._stop.set()
        self._thread.join(timeout=timeout)

    def _run(self):
        while not self._stop.is_set():
            ts = _time.time()
            if PYNVML_OK:
                try:
                    util = pynvml.nvmlDeviceGetUtilizationRates(_nvml_handle).gpu
                    mem_info = pynvml.nvmlDeviceGetMemoryInfo(_nvml_handle)
                    used_mib = mem_info.used / 1024**2
                    self.samples.append((ts, util, used_mib))
                except Exception:
                    pass
            _time.sleep(self.interval)

    @property
    def avg_util(self) -> float:
        utils = [s[1] for s in self.samples]
        return sum(utils) / len(utils) if utils else 0.0

    @property
    def peak_used_mib(self) -> float:
        mems = [s[2] for s in self.samples]
        return max(mems) if mems else 0.0

    def as_dict_list(self) -> list[dict]:
        return [
            {"ts": s[0], "util_pct": s[1], "used_mem_mib": s[2]}
            for s in self.samples
        ]


_poller = GpuPoller(interval=POLL_INTERVAL)
print("[h2-infer] GpuPoller 준비 완료")


In [ ]:
# ══════════════════════════════════════════════════════
# fgpu stderr 로그 음소거 (MUTE_FGPU_LOG 제어)
# ══════════════════════════════════════════════════════
# PYTORCH_NO_CUDA_MEMORY_CACHING=1 이면 generate() 중 수천 줄 쏟아져
# Jupyter 렉 발생. fd 2 를 /dev/null 로 돌려 숨긴다.
# quota enforcement(ALLOW/DENY 판정)에는 영향 없음 — 출력만 끈다.
# memory_idle에서도 resident tensor + matmul 시 ALLOW/FREE 로그 동일하게 적용.
import os, sys

_FGPU_SAVED_STDERR_FD = None
if MUTE_FGPU_LOG:
    sys.stderr.flush()
    _FGPU_SAVED_STDERR_FD = os.dup(2)
    _devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(_devnull, 2)
    os.close(_devnull)
    print("[h2-infer] fgpu stderr 로그 음소거 ON")
    print("[h2-infer] 다시 보려면:  import os; os.dup2(_FGPU_SAVED_STDERR_FD, 2)")
else:
    print("[h2-infer] fgpu stderr 로그 그대로 출력 (MUTE_FGPU_LOG=False)")


In [ ]:
# ══════════════════════════════════════════════════════
# 모델/텐서 로드 — WORKLOAD 분기
# ══════════════════════════════════════════════════════
# compute  : Qwen2-0.5B 모델 로드 (H1과 동일 경로)
# memory_idle: 모델 로드 건너뜀. GPU에 RESIDENT_GIB만큼 큰 텐서 상주 할당.
#              텐서는 float32 (bf16보다 메모리 배분 명확),
#              실험 전체에서 유지 → hook quota 내에서 메모리 점유.
# OOM 처리: try/except — H2에서 큰 텐서+모델 동시면 메모리 경계 가능.
#           throttle OFF·memory ratio는 유지하므로 OOM 시 기록하고 계속.
# ══════════════════════════════════════════════════════
import torch
import os

MODEL_OOM = False
model = None
tokenizer = None
_resident_tensor = None   # memory_idle 상주 텐서 (GC 방지용 전역 참조)

torch.cuda.reset_peak_memory_stats()

if WORKLOAD == "compute":
    # ── LLM 모델 로드 (H1과 동일) ─────────────────────────
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"[h2-infer] WORKLOAD=compute: 모델 로드 시작: {MODEL_ID}")
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.bfloat16,
            device_map="cuda:0",
        )
        model.eval()
        load_peak_mib = torch.cuda.max_memory_allocated() / 1024**2
        print(f"[h2-infer] 모델 로드 완료, peak_mem_allocated={load_peak_mib:.1f} MiB")
        if torch.cuda.is_available():
            total_mib = torch.cuda.get_device_properties(0).total_memory / 1024**2
            quota_mib = total_mib * RATIO
            print(f"[h2-infer] 실사용/quota = {load_peak_mib:.0f}/{quota_mib:.0f} MiB"
                  f" ({load_peak_mib/quota_mib*100:.1f}%)")
    except torch.cuda.OutOfMemoryError as e:
        MODEL_OOM = True
        print(f"[h2-infer] OOM — 모델 로드 실패 (hook DENY 또는 물리 한계): {e}")
    except Exception as e:
        MODEL_OOM = True
        print(f"[h2-infer] 모델 로드 오류: {type(e).__name__}: {e}")

elif WORKLOAD == "memory_idle":
    # ── 상주 텐서 할당 (모델 로드 없음) ──────────────────────
    # RESIDENT_GIB 만큼 GPU에 float32 텐서 상주 → hook quota 내 메모리 점유.
    # 텐서를 _resident_tensor 에 저장해 GC 방지.
    resident_elems = int(RESIDENT_GIB * 1024**3 / 4)  # float32 = 4 bytes
    print(f"[h2-infer] WORKLOAD=memory_idle: 상주 텐서 할당 시작")
    print(f"[h2-infer] RESIDENT_GIB={RESIDENT_GIB}, 요소 수={resident_elems:,}")
    try:
        _resident_tensor = torch.zeros(resident_elems, dtype=torch.float32, device="cuda:0")
        torch.cuda.synchronize()
        alloc_mib = torch.cuda.memory_allocated() / 1024**2
        print(f"[h2-infer] 상주 텐서 할당 완료: {alloc_mib:.1f} MiB allocated")
        if torch.cuda.is_available():
            total_mib = torch.cuda.get_device_properties(0).total_memory / 1024**2
            quota_mib = total_mib * RATIO
            print(f"[h2-infer] 실사용/quota = {alloc_mib:.0f}/{quota_mib:.0f} MiB"
                  f" ({alloc_mib/quota_mib*100:.1f}%)")
        # tokenizer/model은 None 유지 — CSV는 gen_tokens=0으로 기록
    except torch.cuda.OutOfMemoryError as e:
        MODEL_OOM = True
        print(f"[h2-infer] OOM — 상주 텐서 할당 실패 (quota 초과): {e}")
        print(f"[h2-infer] RESIDENT_GIB={RESIDENT_GIB} 를 줄이거나 RATIO를 높이세요.")
    except Exception as e:
        MODEL_OOM = True
        print(f"[h2-infer] 상주 텐서 할당 오류: {type(e).__name__}: {e}")
else:
    print(f"[h2-infer] 알 수 없는 WORKLOAD={WORKLOAD!r}")
    MODEL_OOM = True

if MODEL_OOM:
    print("[h2-infer] 로드/할당 실패로 이후 측정 셀은 건너뜁니다.")


In [ ]:
# ══════════════════════════════════════════════════════
# 시나리오 준비 — 동시 실행 안내
# ══════════════════════════════════════════════════════
# H2는 배리어 동기화 없이 수동 위상차를 사용한다:
#   - A(compute)를 먼저 Run All → warmup(≈30s) 중 B를 Run All
#   - A 본 측정 20회 전체 구간 동안 B가 상주 + burst/sleep 루프 돌아야 함
#   - makespan은 사후 iter_start/end_epoch로 정밀 계산하므로 시작 skew 무관
# 컨테이너 간 /workspace 분리로 파일 플래그 공유 불가 → 수동이 현실적
import time as _time_mod

_model_ready_epoch = _time_mod.time()

if SCENARIO == "h2_sharing":
    print(f"[h2-infer] ★ H2 공유 실험 모드 ★")
    print(f"[h2-infer] WORKLOAD={WORKLOAD}")
    print(f"[h2-infer] 모델/텐서 준비 완료 시각: {_model_ready_epoch:.3f}")
    if WORKLOAD == "compute":
        print(f"[h2-infer] → A(compute): warmup {WARMUP}회 후 본 측정 시작")
        print(f"[h2-infer]   본 측정 시작 직후 B(memory_idle) 세션 Run All 하세요.")
        print(f"[h2-infer]   A 20회 측정(≈{N_ITERS * 6:.0f}~{N_ITERS * 15:.0f}s 예상) 동안 B 상주 필요.")
    elif WORKLOAD == "memory_idle":
        print(f"[h2-infer] → B(memory_idle): 상주 텐서 준비됨, 측정 루프 시작")
        print(f"[h2-infer]   IDLE_SLEEP_S={IDLE_SLEEP_S}s, N_ITERS_IDLE={N_ITERS_IDLE}회")
        print(f"[h2-infer]   총 최소 실행 시간 ≈ {N_ITERS_IDLE * IDLE_SLEEP_S:.0f}s (sleep만)")
elif SCENARIO == "solo":
    print(f"[h2-infer] solo 모드 (WORKLOAD={WORKLOAD}): 기준선 측정.")
    print(f"[h2-infer] H2 기준선은 solo_tps_A(compute)와 B cycle_time(idle)를 기록합니다.")
else:
    print(f"[h2-infer] SCENARIO={SCENARIO!r}. 진행합니다.")

print(f"[h2-infer] 측정 셀로 진행합니다.")


In [ ]:
# ══════════════════════════════════════════════════════
# 측정 루프 — WORKLOAD 분기
# ══════════════════════════════════════════════════════
# compute    : 기존과 동일 — warmup → N_ITERS generate, iter별 타임스탬프 기록
# memory_idle: (a) 상주 텐서는 이미 할당됨
#              (b) N_ITERS_IDLE 루프:
#                  matmul MATMUL_DIM×MATMUL_DIM (1회) + synchronize → 타임스탬프
#                  → iter 밖에서 time.sleep(IDLE_SLEEP_S) (GPU idle 구간)
#              gen_tokens=0, tokens_per_s=0 으로 CSV 스키마 유지
#              latency_s = matmul compute burst 시간 (sleep 제외)
# 두 경로 모두 OOM → try/except 후 기록
# ══════════════════════════════════════════════════════
import time
import torch

_iter_results = []    # (iter_start_epoch, iter_end_epoch, latency_s, gen_tokens, tps)
_meas_start_epoch = time.time()
_meas_end_epoch   = time.time()
_peak_alloc_mib   = 0.0
_peak_reserved_mib = 0.0
_gpu_util_avg     = 0.0
_gpu_util_samples_n = 0
_poller_data      = []

if MODEL_OOM:
    print("[h2-infer] 로드/할당 OOM — 측정 건너뜀")

elif WORKLOAD == "compute":
    # ── compute: LLM generate ─────────────────────────────
    if model is None or tokenizer is None:
        print("[h2-infer] 모델 미로드 — compute 측정 건너뜀")
    else:
        inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda:0")
        prompt_tokens = inputs["input_ids"].shape[-1]
        print(f"[h2-infer] 프롬프트 토큰 수: {prompt_tokens}")
        print(f"[h2-infer] WARMUP={WARMUP}, N_ITERS={N_ITERS}, MAX_NEW_TOKENS={MAX_NEW_TOKENS}")

        # 워밍업
        print("[h2-infer] 워밍업 시작...")
        with torch.no_grad():
            for w in range(WARMUP):
                _ = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )
        torch.cuda.synchronize()
        print(f"[h2-infer] 워밍업 {WARMUP}회 완료. ★ 이 시점 직후 B 세션(memory_idle) Run All 하세요.")

        # 본 측정
        torch.cuda.reset_peak_memory_stats()
        _poller.start()
        _meas_start_epoch = time.time()
        print("[h2-infer] 본 측정 시작...")

        with torch.no_grad():
            for i in range(N_ITERS):
                try:
                    _t0_epoch = time.time()
                    _t0 = time.perf_counter()

                    out = model.generate(
                        **inputs,
                        max_new_tokens=MAX_NEW_TOKENS,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                    )
                    torch.cuda.synchronize()

                    _t1 = time.perf_counter()
                    _t1_epoch = time.time()

                    latency_s = _t1 - _t0
                    gen_tokens = out.shape[-1] - inputs["input_ids"].shape[-1]
                    tps = gen_tokens / latency_s if latency_s > 0 else 0.0
                    _iter_results.append((_t0_epoch, _t1_epoch, latency_s, gen_tokens, tps))

                    if (i + 1) % 5 == 0 or i == 0:
                        print(f"  iter {i+1:3d}/{N_ITERS}: {latency_s:.2f}s,"
                              f" {gen_tokens} tok, {tps:.1f} tok/s")
                except torch.cuda.OutOfMemoryError as e:
                    _t1_epoch = time.time()
                    print(f"  iter {i+1}: OOM — {e}")
                    _iter_results.append((_t0_epoch, _t1_epoch, 0.0, 0, 0.0))

        _meas_end_epoch = time.time()
        torch.cuda.synchronize()
        _poller.stop()
        _peak_alloc_mib    = torch.cuda.max_memory_allocated() / 1024**2
        _peak_reserved_mib = torch.cuda.max_memory_reserved()  / 1024**2
        _gpu_util_avg      = _poller.avg_util
        _gpu_util_samples_n = len(_poller.samples)
        _poller_data       = _poller.as_dict_list()
        print(f"\n[h2-infer] compute 측정 완료 {N_ITERS}회,"
              f" elapsed={_meas_end_epoch - _meas_start_epoch:.2f}s")

elif WORKLOAD == "memory_idle":
    # ── memory_idle: 상주 텐서 + 주기적 matmul burst + sleep ─
    # 상주 텐서는 이미 cell-6에서 _resident_tensor에 할당됨.
    # iter 구조:
    #   iter_start_epoch = time.time() (직전)
    #   matmul MATMUL_DIM×MATMUL_DIM + synchronize (compute burst)
    #   iter_end_epoch = time.time() (burst 직후)
    #   latency_s = burst 시간만 (sleep 제외 → H2 분석에서 cycle_time은 sleep+burst)
    #   time.sleep(IDLE_SLEEP_S) ← iter 밖 (GPU idle 구간)
    #   다음 iter로
    # gen_tokens=0, tokens_per_s=0 (LLM 아님 — CSV 스키마 호환)
    if _resident_tensor is None:
        print("[h2-infer] 상주 텐서 미할당 — memory_idle 측정 건너뜀")
    else:
        # matmul용 입력 텐서 (MATMUL_DIM×MATMUL_DIM, float32)
        # quota 내 메모리 추가 사용: 4×MATMUL_DIM² bytes × 2 (A·B) ≈ 128MB for 4096
        try:
            _mat_a = torch.randn(MATMUL_DIM, MATMUL_DIM, dtype=torch.float32, device="cuda:0")
            _mat_b = torch.randn(MATMUL_DIM, MATMUL_DIM, dtype=torch.float32, device="cuda:0")
        except torch.cuda.OutOfMemoryError as e:
            MODEL_OOM = True
            print(f"[h2-infer] matmul 입력 텐서 할당 OOM: {e}")
            print(f"[h2-infer] MATMUL_DIM={MATMUL_DIM} 또는 RESIDENT_GIB={RESIDENT_GIB} 를 줄이세요.")

        if not MODEL_OOM:
            torch.cuda.reset_peak_memory_stats()
            _poller.start()
            _meas_start_epoch = time.time()
            print(f"[h2-infer] memory_idle 측정 시작: N_ITERS_IDLE={N_ITERS_IDLE}회")
            print(f"[h2-infer] MATMUL_DIM={MATMUL_DIM}, IDLE_SLEEP_S={IDLE_SLEEP_S}s")
            print(f"[h2-infer] 총 최소 시간 ≈ {N_ITERS_IDLE * IDLE_SLEEP_S:.0f}s")

            for i in range(N_ITERS_IDLE):
                try:
                    _t0_epoch = time.time()
                    _t0 = time.perf_counter()

                    # compute burst: matmul 1회 + synchronize
                    _ = torch.mm(_mat_a, _mat_b)
                    torch.cuda.synchronize()

                    _t1 = time.perf_counter()
                    _t1_epoch = time.time()

                    latency_s = _t1 - _t0   # burst 시간만
                    # gen_tokens=0, tokens_per_s=0 (LLM 아님 — CSV 스키마 호환)
                    _iter_results.append((_t0_epoch, _t1_epoch, latency_s, 0, 0.0))

                    if (i + 1) % 5 == 0 or i == 0:
                        print(f"  iter {i+1:3d}/{N_ITERS_IDLE}: burst={latency_s*1000:.1f}ms"
                              f" → sleep {IDLE_SLEEP_S}s ...")

                    # sleep: iter 밖에서 GPU를 비움 (idle fraction 조절 핵심)
                    time.sleep(IDLE_SLEEP_S)

                except torch.cuda.OutOfMemoryError as e:
                    _t1_epoch = time.time()
                    print(f"  iter {i+1}: OOM — {e}")
                    _iter_results.append((_t0_epoch, _t1_epoch, 0.0, 0, 0.0))
                    time.sleep(IDLE_SLEEP_S)

            _meas_end_epoch = time.time()
            torch.cuda.synchronize()
            _poller.stop()
            _peak_alloc_mib    = torch.cuda.max_memory_allocated() / 1024**2
            _peak_reserved_mib = torch.cuda.max_memory_reserved()  / 1024**2
            _gpu_util_avg      = _poller.avg_util
            _gpu_util_samples_n = len(_poller.samples)
            _poller_data       = _poller.as_dict_list()
            print(f"\n[h2-infer] memory_idle 측정 완료 {N_ITERS_IDLE}회,"
                  f" elapsed={_meas_end_epoch - _meas_start_epoch:.2f}s")
else:
    print(f"[h2-infer] 알 수 없는 WORKLOAD={WORKLOAD!r} — 측정 건너뜀")


In [ ]:
# ══════════════════════════════════════════════════════
# 통계 계산
# ══════════════════════════════════════════════════════
# compute  : mean/p50/p95 latency, mean tps, total gen_tokens
# memory_idle: mean latency(=burst 시간), tps=0, gen_tokens=0
#              cycle_time(solo) 는 메타 JSON에 idle_sleep_s + mean_latency_s 로 기록
import statistics

def _pct(data, p):
    """선형 보간 백분위수."""
    if not data:
        return 0.0
    sdata = sorted(data)
    idx = (len(sdata) - 1) * p / 100
    lo, hi = int(idx), min(int(idx) + 1, len(sdata) - 1)
    return sdata[lo] + (sdata[hi] - sdata[lo]) * (idx - lo)

if not _iter_results:
    _stats = {
        "oom": True,
        "n_iters": 0,
        "makespan_s": 0.0,
        "mean_latency_s": 0.0,
        "p50_latency_s": 0.0,
        "p95_latency_s": 0.0,
        "mean_tokens_per_s": 0.0,
        "total_gen_tokens": 0,
        "t_start_epoch": _meas_start_epoch,
        "t_end_epoch": _meas_end_epoch,
    }
else:
    latencies = [r[2] for r in _iter_results]
    tps_list  = [r[4] for r in _iter_results]
    gen_toks  = [r[3] for r in _iter_results]
    n = len(_iter_results)

    _stats = {
        "oom": False,
        "n_iters": n,
        "makespan_s": _meas_end_epoch - _meas_start_epoch,
        "mean_latency_s":    statistics.mean(latencies) if latencies else 0.0,
        "p50_latency_s":     _pct(latencies, 50),
        "p95_latency_s":     _pct(latencies, 95),
        "mean_tokens_per_s": statistics.mean(tps_list) if tps_list else 0.0,
        "total_gen_tokens":  sum(gen_toks),
        "t_start_epoch":     _meas_start_epoch,
        "t_end_epoch":       _meas_end_epoch,
    }

print("[h2-infer] 통계 계산 완료")
if WORKLOAD == "memory_idle" and _iter_results:
    # cycle_time = sleep + burst (solo 기준선으로 사용)
    mean_burst = _stats["mean_latency_s"]
    cycle_time = mean_burst + IDLE_SLEEP_S
    print(f"[h2-infer] memory_idle 통계:")
    print(f"  mean burst latency = {mean_burst*1000:.1f} ms")
    print(f"  IDLE_SLEEP_S       = {IDLE_SLEEP_S} s")
    print(f"  cycle_time (burst+sleep) = {cycle_time:.3f} s")
    idle_frac = IDLE_SLEEP_S / cycle_time if cycle_time > 0 else 0.0
    print(f"  idle fraction      = {idle_frac*100:.1f}%")


In [ ]:
# ══════════════════════════════════════════════════════
# 결과 저장
# ══════════════════════════════════════════════════════
# CSV 컬럼: label,workload,scenario,ratio,model,iter,
#           iter_start_epoch,iter_end_epoch,latency_s,gen_tokens,tokens_per_s
# workload 컬럼 추가 (H1과 차이점, 분석 노트북에서 필터용)
# 파일명에 SCENARIO 포함 → 같은 세션에서 여러 시나리오 가능
# ══════════════════════════════════════════════════════
import csv, json, os, time as _time_save

RESULT_CSV = f"/workspace/session_result_{SCENARIO}.csv"
META_JSON  = f"/workspace/session_meta_{SCENARIO}.json"

# ── CSV 저장 ──────────────────────────────────────────
csv_header = [
    "label", "workload", "scenario", "ratio", "model",
    "iter", "iter_start_epoch", "iter_end_epoch",
    "latency_s", "gen_tokens", "tokens_per_s",
]

# memory_idle에서 모델 ID 대신 워크로드 설명 기록 (분석 노트북 편의)
_model_field = MODEL_ID if WORKLOAD == "compute" else f"memory_idle_dim{MATMUL_DIM}"

with open(RESULT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(csv_header)
    if _iter_results:
        for idx, (t0e, t1e, lat, gtok, tps) in enumerate(_iter_results):
            writer.writerow([
                SESSION_LABEL,
                WORKLOAD,
                SCENARIO,
                RATIO,
                _model_field,
                idx + 1,
                f"{t0e:.6f}",
                f"{t1e:.6f}",
                f"{lat:.6f}",
                gtok,
                f"{tps:.4f}",
            ])
    else:
        writer.writerow([
            SESSION_LABEL, WORKLOAD, SCENARIO, RATIO, _model_field,
            0, f"{_meas_start_epoch:.6f}", f"{_meas_end_epoch:.6f}",
            0.0, 0, 0.0,
        ])

print(f"[h2-infer] CSV 저장: {RESULT_CSV}")

# ── 메타 JSON 저장 ────────────────────────────────────
env_snap = {
    k: os.environ.get(k, None)
    for k in [
        "PYTORCH_NO_CUDA_MEMORY_CACHING",
        "LD_PRELOAD",
        "FGPU_RATIO",
        "FGPU_QUOTA_BYTES",
        "FGPU_THROTTLE_ENABLE",
        "FGPU_COMPUTE_RATIO",
        "FGPU_LAUNCH_LOG_EVERY",
    ]
}

meta = {
    "label":    SESSION_LABEL,
    "workload": WORKLOAD,
    "scenario": SCENARIO,
    "ratio":    RATIO,
    "model":    _model_field,
    "oom":      MODEL_OOM or _stats.get("oom", False),
    "n_iters":  _stats["n_iters"],
    "makespan_s":       _stats["makespan_s"],
    "t_start_epoch":    _stats["t_start_epoch"],
    "t_end_epoch":      _stats["t_end_epoch"],
    "mean_latency_s":   _stats["mean_latency_s"],
    "p50_latency_s":    _stats["p50_latency_s"],
    "p95_latency_s":    _stats["p95_latency_s"],
    "mean_tokens_per_s": _stats["mean_tokens_per_s"],
    "total_gen_tokens": _stats["total_gen_tokens"],
    "peak_mem_alloc_mib":    _peak_alloc_mib,
    "peak_mem_reserved_mib": _peak_reserved_mib,
    "gpu_util_avg_pct":   _gpu_util_avg,
    "gpu_util_samples_n": _gpu_util_samples_n,
    "gpu_util_timeline":  _poller_data[:500],
    "env_snapshot": env_snap,
    "saved_at": _time_save.strftime("%Y-%m-%dT%H:%M:%SZ", _time_save.gmtime()),
}

# memory_idle 전용 필드 추가
if WORKLOAD == "memory_idle":
    meta["resident_gib"]  = RESIDENT_GIB
    meta["matmul_dim"]    = MATMUL_DIM
    meta["idle_sleep_s"]  = IDLE_SLEEP_S
    meta["n_iters_idle"]  = N_ITERS_IDLE
    if _iter_results:
        mean_burst = _stats["mean_latency_s"]
        cycle_time = mean_burst + IDLE_SLEEP_S
        idle_frac  = IDLE_SLEEP_S / cycle_time if cycle_time > 0 else 0.0
        meta["cycle_time_s"]  = cycle_time
        meta["idle_fraction"] = idle_frac
        meta["cycle_times"]   = [r[2] + IDLE_SLEEP_S for r in _iter_results]

with open(META_JSON, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[h2-infer] 메타 저장: {META_JSON}")


In [ ]:
# ══════════════════════════════════════════════════════
# 요약 출력 — 분석 노트북 없이도 한눈에 확인
# ══════════════════════════════════════════════════════
import time as _t_sum, os

print("=" * 64)
print(f"  fGPU H2 추론 실험 요약")
print(f"  세션: {SESSION_LABEL}  워크로드: {WORKLOAD}")
print(f"  시나리오: {SCENARIO}  RATIO: {RATIO}")
print("=" * 64)

if MODEL_OOM or not _iter_results:
    print("  상태: OOM / 측정 실패")
    print(f"  파일: {RESULT_CSV}")
    print(f"        {META_JSON}")
else:
    print(f"  반복 횟수    : {_stats['n_iters']} 회")
    print(f"  총 경과 시간 : {_stats['makespan_s']:.2f} s")
    print(f"  mean latency : {_stats['mean_latency_s']:.3f} s")
    print(f"  p50  latency : {_stats['p50_latency_s']:.3f} s")
    print(f"  p95  latency : {_stats['p95_latency_s']:.3f} s")

    if WORKLOAD == "compute":
        print(f"  mean tok/s   : {_stats['mean_tokens_per_s']:.2f}")
        print(f"  총 gen tok   : {_stats['total_gen_tokens']}")
        print()
        print(f"  [H2 성공 기준 (사후 분석)]")
        print(f"  1순위: overlap_tps_A / solo_tps_A >= 0.80")
        print(f"  2순위: speedup(overlap) >= 1.1 (stretch: 1.2)")
        print(f"  → fgpu_analysis.ipynb 로 solo vs h2_sharing 비교")
    elif WORKLOAD == "memory_idle":
        if _iter_results:
            mean_burst = _stats["mean_latency_s"]
            cycle_time = mean_burst + IDLE_SLEEP_S
            idle_frac  = IDLE_SLEEP_S / cycle_time if cycle_time > 0 else 0.0
            print(f"  mean burst   : {mean_burst*1000:.1f} ms")
            print(f"  cycle_time   : {cycle_time:.3f} s")
            print(f"  idle fraction: {idle_frac*100:.1f}%")
            print()
            print(f"  [pynvml 타임라인] avg_util={_gpu_util_avg:.1f}%"
                  f" ({_gpu_util_samples_n} samples)")
            print(f"  → burst/sleep 골 패턴 확인: session_meta_{SCENARIO}.json['gpu_util_timeline']")

    print()
    if torch.cuda.is_available():
        total_mib = torch.cuda.get_device_properties(0).total_memory / 1024**2
        quota_mib = total_mib * RATIO
        print(f"  peak mem alloc : {_peak_alloc_mib:.1f} MiB"
              f" ({_peak_alloc_mib/quota_mib*100:.1f}% of quota {quota_mib:.0f} MiB)")
        print(f"  peak mem rsv   : {_peak_reserved_mib:.1f} MiB")

print()
print(f"  결과 파일: {RESULT_CSV}")
print(f"  메타 파일: {META_JSON}")
print("=" * 64)
print("  다음: fgpu_analysis.ipynb 로 solo vs h2_sharing 비교 분석")
print("=" * 64)
